# Evaluasi Kuantitatif Chatbot (LLM-as-a-judge)
Notebook ini mengambil data dari Supabase (history chat) dan SQLite (progress user materi), lalu menggunakan Gemini API sebagai juri penilai.

In [ ]:
import os
import sqlite3
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client, Client
import google.generativeai as genai
import json
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score

# Load env vars from backend folder
load_dotenv('../backend/.env')

SUPABASE_URL = os.environ.get('SUPABASE_URL')
SUPABASE_KEY = os.environ.get('SUPABASE_SERVICE_ROLE_KEY') or os.environ.get('SUPABASE_ANON_KEY')
GEMINI_API_KEY = os.environ.get('LEVELY_GEMINI_API_KEY')

if not SUPABASE_URL or not SUPABASE_KEY:
    print('Missing Supabase credentials in ../backend/.env')

try:
    supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    genai.configure(api_key=GEMINI_API_KEY)
except Exception as e:
    print('Failed to setup clients:', e)

## 1. Ambil Data Percakapan dari Supabase

In [ ]:
try:
    # Ambil semua sesi
    response = supabase.table('chat_sessions').select('*').execute()
    sessions = response.data
    df_sessions = pd.DataFrame(sessions)

    # Ambil semua pesan
    msg_response = supabase.table('chat_messages').select('*').execute()
    messages = msg_response.data
    df_messages = pd.DataFrame(messages)

    print(f"Total sessions: {len(df_sessions)}")
    print(f"Total messages: {len(df_messages)}")
    display(df_messages.head())
except Exception as e:
    print('Supabase fetch error:', e)
    df_messages = pd.DataFrame()
    df_sessions = pd.DataFrame()

## 2. Ambil Data Context User & Materi dari SQLite (Prisma)
Karena di ChatbotController kita tahu ada `materialId` dan info `User`, kita akan coba mengambil konteks terkait.

In [ ]:
DB_PATH = '../backend/your_database_file.db'
try:
    conn = sqlite3.connect(DB_PATH)

    # Fetch Users info (Points, name)
    df_users = pd.read_sql_query('SELECT id, name, username, points FROM User', conn)
    conn.close()

    # Gabungkan data sesi dengan pesan untuk dapatkan user_id
    if not df_messages.empty and not df_sessions.empty:
        df = pd.merge(df_messages, df_sessions[['id', 'user_id']], left_on='session_id', right_on='id', suffixes=('', '_session'))
        df = pd.merge(df, df_users, left_on='user_id', right_on='id', how='left')
        display(df.head())
    else:
        df = pd.DataFrame()
except Exception as e:
    print('Database error:', e)
    df = pd.DataFrame()

## 3. Pemrosesan Pasangan (Prompt User - Balasan Chatbot)
Untuk mengevaluasi chatbot, kita perlu menyandingkan isi pesan dari `user` dan balasannya dari `assistant`.

In [ ]:
pairs = []
if not df.empty:
    # Urutkan pesan berdasarkan sesi dan waktu dibuat
    df = df.sort_values(['session_id', 'created_at'])

    current_user_msg = None

    for idx, row in df.iterrows():
        if row['role'] == 'user':
            current_user_msg = row
        elif row['role'] == 'assistant' and current_user_msg is not None:
            # Kita dapatkan pasangan
            pairs.append({
                'session_id': row['session_id'],
                'user_id': row['user_id'],
                'user_points': row.get('points', 0),
                'user_message': current_user_msg['content'],
                'chatbot_reply': row['content'],
                'created_at': row['created_at']
            })
            current_user_msg = None

df_pairs = pd.DataFrame(pairs)
print(f"Total user-assistant interaction pairs: {len(df_pairs)}")
if not df_pairs.empty:
    display(df_pairs.head())

## 4. Evaluasi Menggunakan LLM-as-a-judge (Gemini Pro)
Kita buat prompt ketat untuk meminta model LLM menilai setiap respons Chatbot dari skala 1-5 berdasarkan seberapa paham dia pada konteks user.

In [ ]:
generation_config = {
  "temperature": 0,
  "response_mime_type": "application/json",
}
try:
    model = genai.GenerativeModel("gemini-1.5-flash", generation_config=generation_config)
except Exception as e:
    print('GenAI model initialization error:', e)

eval_prompt = """
Kamu adalah juri evaluator percakapan chatbot pendidikan bernama Levely.
Tugasmu adalah memberikan skor 1 hingga 5 pada balasan chatbot Levely berdasarkan pesan terakhir dari user.
Kriteria Penilaian:
- Skor 1: Balasan sama sekali tidak masuk akal, salah kaprah, atau menolak menjawab tanpa alasan.
- Skor 2: Balasan kurang relevan dan tidak berusaha memotivasi.
- Skor 3: Balasan standar (cukup relevan), namun kaku atau kurang personal.
- Skor 4: Balasan bagus, relevan dengan materi, dan cukup memotivasi dengan bahasa yang ramah.
- Skor 5: Balasan sangat luar biasa, sangat pas menjawab pertanyaan secara rinci sekaligus memperhatikan progress/dorongan positif ke siswa.

Beri output dalam format JSON ketat:
{
  "score": <angka_1_sampai_5>,
  "reason": "<alasan singkat max 1 kalimat>"
}

Konteks Interaksi:
- User (Point/Skor: {user_points}): {user_message}
- Chatbot Levely: {chatbot_reply}
"""

def evaluate_with_llm(row):
    prompt = eval_prompt.format(
        user_points=row['user_points'],
        user_message=row['user_message'],
        chatbot_reply=row['chatbot_reply']
    )
    try:
        response = model.generate_content(prompt)
        res_json = json.loads(response.text)
        # Handle API rate limit by putting slight delay
        time.sleep(1)
        return pd.Series([res_json.get('score'), res_json.get('reason')])
    except Exception as e:
        time.sleep(1)
        return pd.Series([None, str(e)])

# JANGAN run loop panjang di production kecuali pakai delay untuk limit API.
# Kita ambil 50 sampel acak untuk tahap awal.
if not df_pairs.empty:
    df_sample = df_pairs.sample(min(50, len(df_pairs)), random_state=42).copy()
    # Uncomment baris ini untuk memulai evaluasi (membutuhkan API Key)
    # df_sample[['llm_score', 'llm_reason']] = df_sample.apply(evaluate_with_llm, axis=1)
    # display(df_sample.head())
else:
    df_sample = pd.DataFrame()

## 5. Ekspor untuk Human Evaluation (Hitung Agreement)
Kita ekspor dataset ini TANPA menyertakan `llm_score` agar rater manusia dapat memberi skor secara manual.

In [ ]:
if not df_sample.empty:
    # Membuat dan menyimpan sample untuk evaluasi manusia
    df_export = df_sample[['session_id', 'user_message', 'chatbot_reply']].copy()
    df_export['human_score'] = ""
    try:
        df_export.to_excel('human_evaluation_sample.xlsx', index=False)
        print("Berhasil diekspor ke human_evaluation_sample.xlsx")
        # df_sample.to_csv('llm_evaluation_backup.csv', index=False) # Simpan backup setelah dievaluasi
    except Exception as e:
        print('Failed to export:', e)

## 6. Hitung Validasi (Cohen's Kappa & Pearson Correlation)
Setelahi rater manusia (anda) mengisi nilai 1-5 di file `human_evaluation_sample.xlsx`, silahkan *load* kembali file tersebut.

In [ ]:
"""
# Uncomment dan jalankan script ini jika Excel sudah diisi:
df_human = pd.read_excel('human_evaluation_sample.xlsx')
df_merged = pd.merge(df_sample, df_human[['session_id', 'user_message', 'human_score']], on=['session_id', 'user_message'])
df_merged = df_merged.dropna(subset=['llm_score', 'human_score'])

# 1. Pearson Correlation
corr = df_merged['llm_score'].astype(float).corr(df_merged['human_score'].astype(float))
print(f"Pearson Correlation (Human vs LLM): {corr:.3f}")

# 2. Kesepakatan Kasar (Exact Match)
exact_match = (df_merged['llm_score'].astype(float) == df_merged['human_score'].astype(float)).mean()
print(f"Tingkat Kesepakatan (Exact Match): {exact_match:.2%}")

# 3. Rata-rata Skor Chatbot Keseluruhan (dari LLM)
avg_score = df_sample['llm_score'].astype(float).mean()
print(f"Rata-rata Skor Kualitas Respons Chatbot: {avg_score:.2f} dari 5.0")

# Visualisasi Distribusi Skor LLM
sns.countplot(data=df_sample, x='llm_score')
plt.title('Distribusi Penilaian Kualitas Respons Chatbot oleh LLM Evaluator')
plt.xlabel('Skor (1-5)')
plt.ylabel('Jumlah Percakapan')
plt.show()
"""
pass